# gProfiler — API & bulk

[g:Profiler](https://biit.cs.ut.ee/gprofiler/) (University of Tartu)
provides functional enrichment plus a per-organism combined gene-set
library (GO, Reactome, KEGG, miRTarBase, TF, HPA, …).

| Mode | Function |
|---|---|
| **REST enrichment** | `gost` |
| **Bulk GMT** | `download_gmt`, `load_gmt` |

The official client (`gprofiler-official`) is preferred; if not
installed, `gost` falls back to a plain `requests.post`.

## 1. REST enrichment — `gost`

Send a list of gene symbols, get back an enrichment DataFrame with the
documented `source` / `name` / `p_value` columns plus all the per-row
metadata `g:Profiler` provides.

In [1]:
from biodb import gprofiler

In [2]:
result = gprofiler.gost(query=["BRCA1", "BRCA2", "TP53"], organism="hsapiens")
print(result.shape)
result[["source", "name", "p_value", "intersection_size", "term_size"]].head(10)

(159, 14)


,source,name,p_value,intersection_size,term_size
0,HP,Extrahepatic cholestasis,0.000004,3,10
1,HP,Functional intestinal obstruction,0.000005,3,11
2,HP,Peritoneal abscess,0.000007,3,12
3,HP,Ovarian carcinoma,0.000011,3,14
4,HP,Primary peritoneal carcinoma,0.000014,3,15
5,WP,DNA IR double strand breaks and cellular respo...,0.000016,3,55
6,GO:BP,cellular response to ionizing radiation,0.000017,3,70
7,WP,Glioblastoma signaling,0.000055,3,82
8,WP,DNA IR damage and cellular response via ATR,0.000055,3,82
9,GO:BP,intrinsic apoptotic signaling pathway in respo...,0.000062,3,107


By default `gost` post-filters to `p_value < 0.05` — pass
`user_threshold=` to relax.

## 2. Bulk — the combined per-organism GMT

`download_gmt(organism=...)` fetches a single ~40 MB
`gprofiler_full_{organism}.name.gmt` from Tartu and caches it under
`~/.cache/biodb/gprofiler/`.

The URL changed after Tartu's 2026 SPA migration — `bioDB` already
points at the new (plain `.gmt`, no zip) endpoint.

In [3]:
print(gprofiler.GPROFILER_GMT_URL_TEMPLATE)

https://biit.cs.ut.ee/gprofiler/static/gprofiler_full_{organism}.name.gmt


```python
path = gprofiler.download_gmt(organism="hsapiens")  # ~40 MB
df = gprofiler.load_gmt(return_format="pandas")     # long DataFrame
d  = gprofiler.load_gmt(return_format="dict")       # {(id, label): [genes]}
```

Once cached, every subsequent call is instant.